# 03 — Extracción del mercado de referencia desde TMDB

Este notebook construye un primer benchmark de **500 películas únicas**. La unidad de análisis es una película, no una reseña ni un campo de la API. Los filtros se definieron antes de observar los resultados para evitar seleccionar casos convenientes.

## Filtros acordados

- Tipo: películas de TMDB (`discover/movie`).
- Estreno principal: cada año entre 2023 y 2025.
- Excluir contenido explícito marcado por TMDB (`include_adult=false`).
- Excluir resultados de video (`include_video=false`).
- Al menos 100 votos TMDB, para que una valoración alta tenga una base mínima de opiniones.
- Sin restricción de género, país o idioma original.
- Selección de las 200 mejor valoradas de cada año, según el rating de TMDB.

Estos filtros no excluyen películas para adolescentes o adultos; la clasificación etaria se analizará después, si está disponible. La muestra final tiene 600 películas: 200 de cada año.

In [8]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import time

import pandas as pd
import requests
from dotenv import load_dotenv


In [9]:
def encontrar_raiz_proyecto(inicio: Path) -> Path:
    for carpeta in (inicio, *inicio.parents):
        if (carpeta / '.env.example').exists():
            return carpeta
    raise FileNotFoundError('No se encontró .env.example en las carpetas superiores.')

RAIZ_PROYECTO = encontrar_raiz_proyecto(Path.cwd().resolve())
load_dotenv(RAIZ_PROYECTO / '.env', override=True)
TMDB_API_KEY = os.getenv('TMDB_API_KEY')

if not TMDB_API_KEY:
    raise ValueError('Falta TMDB_API_KEY en el archivo local .env.')

print('Clave de TMDB detectada localmente.')

Clave de TMDB detectada localmente.


In [10]:
BASE_URL_TMDB = 'https://api.themoviedb.org/3'
FILTROS_BASE = {
    'include_adult': 'false',
    'include_video': 'false',
    'vote_count.gte': 100,
    'sort_by': 'vote_average.desc',
    'language': 'es-AR',
}
CUOTAS_POR_ANIO = {2023: 200, 2024: 200, 2025: 200}
CANTIDAD_OBJETIVO = sum(CUOTAS_POR_ANIO.values())

def consultar_descubrimiento(anio: int, pagina: int) -> dict:
    respuesta = requests.get(
        f'{BASE_URL_TMDB}/discover/movie',
        params={
            'api_key': TMDB_API_KEY,
            **FILTROS_BASE,
            'primary_release_date.gte': f'{anio}-01-01',
            'primary_release_date.lte': f'{anio}-12-31',
            'page': pagina,
        },
        timeout=30,
    )
    respuesta.raise_for_status()
    return respuesta.json()

In [11]:
peliculas_crudas = []
paginas_consultadas = {}

for anio, cuota in CUOTAS_POR_ANIO.items():
    peliculas_anio = []
    paginas_consultadas[anio] = []
    pagina = 1
    total_paginas = None

    while len(peliculas_anio) < cuota:
        resultado = consultar_descubrimiento(anio, pagina)
        paginas_consultadas[anio].append(pagina)
        peliculas_anio.extend(resultado['results'])
        total_paginas = resultado['total_pages']
        print(f'Año {anio}, página {pagina}/{total_paginas}: acumulado {len(peliculas_anio)}')

        if pagina >= total_paginas:
            break
        pagina += 1
        time.sleep(0.35)

    muestra_anio = pd.DataFrame(peliculas_anio).drop_duplicates(subset='id').head(cuota)
    if len(muestra_anio) < cuota:
        raise ValueError(f'En {anio} solo se obtuvieron {len(muestra_anio)} películas; revisar el mínimo de votos.')
    peliculas_crudas.extend(muestra_anio.to_dict(orient='records'))

mercado_crudo = pd.DataFrame(peliculas_crudas).drop_duplicates(subset='id').copy()

if len(mercado_crudo) != CANTIDAD_OBJETIVO:
    raise ValueError(f'Se esperaban {CANTIDAD_OBJETIVO} películas y se obtuvieron {len(mercado_crudo)}.')

print(f'Películas únicas obtenidas: {len(mercado_crudo)}')

Año 2023, página 1/36: acumulado 20
Año 2023, página 2/36: acumulado 40
Año 2023, página 3/36: acumulado 60
Año 2023, página 4/36: acumulado 80
Año 2023, página 5/36: acumulado 100
Año 2023, página 6/36: acumulado 120
Año 2023, página 7/36: acumulado 140
Año 2023, página 8/36: acumulado 160
Año 2023, página 9/36: acumulado 180
Año 2023, página 10/36: acumulado 200
Año 2024, página 1/30: acumulado 20
Año 2024, página 2/30: acumulado 40
Año 2024, página 3/30: acumulado 60
Año 2024, página 4/30: acumulado 80
Año 2024, página 5/30: acumulado 100
Año 2024, página 6/30: acumulado 120
Año 2024, página 7/30: acumulado 140
Año 2024, página 8/30: acumulado 160
Año 2024, página 9/30: acumulado 180
Año 2024, página 10/30: acumulado 200
Año 2025, página 1/25: acumulado 20
Año 2025, página 2/25: acumulado 40
Año 2025, página 3/25: acumulado 60
Año 2025, página 4/25: acumulado 80
Año 2025, página 5/25: acumulado 100
Año 2025, página 6/25: acumulado 120
Año 2025, página 7/25: acumulado 140
Año 2025, p

In [12]:
peliculas_mercado = mercado_crudo.rename(columns={
    'id': 'tmdb_id',
    'title': 'titulo',
    'original_title': 'titulo_original',
    'release_date': 'fecha_estreno',
    'original_language': 'idioma_original',
    'genre_ids': 'genero_ids_tmdb',
    'popularity': 'popularidad_tmdb',
    'vote_average': 'rating_tmdb',
    'vote_count': 'votos_tmdb',
    'adult': 'marcado_adulto_tmdb',
    'video': 'marcado_video_tmdb',
})

columnas_csv = [
    'tmdb_id', 'titulo', 'titulo_original', 'fecha_estreno', 'idioma_original',
    'genero_ids_tmdb', 'popularidad_tmdb', 'rating_tmdb', 'votos_tmdb',
    'marcado_adulto_tmdb', 'marcado_video_tmdb',
]
peliculas_mercado = peliculas_mercado[columnas_csv].copy()
peliculas_mercado['fecha_estreno'] = pd.to_datetime(peliculas_mercado['fecha_estreno'], errors='coerce')
peliculas_mercado['genero_ids_tmdb'] = peliculas_mercado['genero_ids_tmdb'].apply(
    lambda generos: '|'.join(map(str, generos)) if isinstance(generos, list) else ''
)

peliculas_mercado.head()

,tmdb_id,titulo,titulo_original,fecha_estreno,idioma_original,genero_ids_tmdb,popularidad_tmdb,rating_tmdb,votos_tmdb,marcado_adulto_tmdb,marcado_video_tmdb
0,1031774,Thriller 40,Thriller 40,2023-12-02,en,99|10402,1.2134,8.700,129,False,False
1,1181678,¿Quieres ser mi... hijo?,¿Quieres ser mi hijo?,2023-09-21,en,35|10749,5.0770,8.400,378,False,False
2,1175306,Me vuelves loca,Me vuelves loca,2023-09-07,es,35,0.7250,8.422,129,False,False
3,1150394,La Leyenda de los Chaneques,La Leyenda de los Chaneques,2023-07-14,es,16|35|10751|27,16.8983,8.383,287,False,False
4,1063453,BTS: Yet to Come in Cinemas,방탄소년단: 옛 투 컴 인 시네마,2023-02-01,ko,10402|99,1.0859,8.371,124,False,False


In [13]:
fecha_extraccion = datetime.now(timezone.utc).date().isoformat()
ruta_json = RAIZ_PROYECTO / 'data' / 'raw' / 'tmdb' / f'mercado_mejor_valorado_2023_2025_{fecha_extraccion}.json'
ruta_csv = RAIZ_PROYECTO / 'data' / 'processed' / 'peliculas_mejor_valoradas_2023_2025.csv'
ruta_metadata = RAIZ_PROYECTO / 'data' / 'raw' / f'metadata_mercado_mejor_valorado_2023_2025_{fecha_extraccion}.json'

ruta_json.parent.mkdir(parents=True, exist_ok=True)
ruta_csv.parent.mkdir(parents=True, exist_ok=True)

with ruta_json.open('w', encoding='utf-8') as archivo:
    json.dump(peliculas_crudas, archivo, ensure_ascii=False, indent=2)

peliculas_mercado.to_csv(ruta_csv, index=False, encoding='utf-8')

metadata = {
    'consulted_at_utc': datetime.now(timezone.utc).isoformat(),
    'source': 'TMDB /discover/movie',
    'base_filters': FILTROS_BASE,
    'pages_consulted': paginas_consultadas,
    'rows_per_year': CUOTAS_POR_ANIO,
    'target_rows': CANTIDAD_OBJETIVO,
    'unique_rows_exported': len(peliculas_mercado),
    'notes': 'Muestra de las películas mejor valoradas por TMDB, con al menos 100 votos y 200 filas por año. Aún no incorpora presupuesto ni recaudación por película.',
}
with ruta_metadata.open('w', encoding='utf-8') as archivo:
    json.dump(metadata, archivo, ensure_ascii=False, indent=2)

print(f'JSON crudo: {ruta_json.relative_to(RAIZ_PROYECTO)}')
print(f'CSV de mercado: {ruta_csv.relative_to(RAIZ_PROYECTO)}')
print(f'Metadatos: {ruta_metadata.relative_to(RAIZ_PROYECTO)}')

JSON crudo: data\raw\tmdb\mercado_mejor_valorado_2023_2025_2026-11-10.json
CSV de mercado: data\processed\peliculas_mejor_valoradas_2023_2025.csv
Metadatos: data\raw\metadata_mercado_mejor_valorado_2023_2025_2026-11-10.json


In [14]:
validacion = {
    'filas': len(peliculas_mercado),
    'ids_unicos': peliculas_mercado['tmdb_id'].nunique(),
    'marcadas_adultas': int(peliculas_mercado['marcado_adulto_tmdb'].sum()),
    'marcadas_como_video': int(peliculas_mercado['marcado_video_tmdb'].sum()),
    'sin_fecha_estreno': int(peliculas_mercado['fecha_estreno'].isna().sum()),
    'votos_tmdb_minimos': int(peliculas_mercado['votos_tmdb'].min()),
    'filas_2023': int((peliculas_mercado['fecha_estreno'].dt.year == 2023).sum()),
    'filas_2024': int((peliculas_mercado['fecha_estreno'].dt.year == 2024).sum()),
    'filas_2025': int((peliculas_mercado['fecha_estreno'].dt.year == 2025).sum()),
}
pd.Series(validacion, name='resultado')

filas                  600
ids_unicos             600
marcadas_adultas         0
marcadas_como_video      0
sin_fecha_estreno        0
votos_tmdb_minimos     100
filas_2023             200
filas_2024             200
filas_2025             200
Name: resultado, dtype: int64

## Límite de esta etapa

El CSV resultante tiene 600 películas: las 200 mejor valoradas de cada año, dentro de los filtros acordados. No contiene todavía presupuesto ni recaudación: TMDB entrega esos campos en el detalle individual de cada película. La próxima etapa deberá enriquecer los registros sin modificar ni perder esta extracción base.